# 02 Execution Parameter Stability Map

Select a signal source and execution components, then search for parameter neighborhoods that remain useful around nearby grid points.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    candidates = [Path.cwd().resolve(), Path('Z:/SEN05_Autotrading'), Path('//10.11.12.6/Share/SEN05_Autotrading')]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / 'pyproject.toml').exists() and (current / 'backtest_optimize').exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError('Could not find project root.')

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
BACKTEST_ROOT = project_root / 'backtest_optimize'
RAW_SIGNALS = project_root / 'raw_signals'
OUTPUT_DIR = BACKTEST_ROOT / 'outputs' / 'stability_maps'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.analysis.notebook_dashboard import (
    build_execution_config, component_catalog_frame, control_panel_frame,
    discover_signal_catalog, run_context_frame, select_signal, style_report,
)
from backtest_optimize.analysis.optimize import build_refined_grid, grid_size
from backtest_optimize.analysis.research_pipeline import (
    evaluate_stability_map, rank_stability_candidates,
)
from backtest_optimize.analysis.versioning import make_run_id, save_snapshot

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 220)

In [ ]:
# Signal source
SIGNAL_STRATEGY = 'combo'
SIGNAL_CHOICE = 'auto'
PREFERRED_SYMBOL = 'US30'
PREFERRED_TIMEFRAME = 'H4'
WARMUP_BARS = 0

# Fixed execution structure. Grid values override only selected parameters.
BASE_CONFIG = build_execution_config(
    engine='component', profile_name='combo_ctrader_v0',
    account_size=10_000.0, risk_per_cluster=0.01,
    ambiguity_policy=AmbiguityPolicy.CONSERVATIVE,
    entry_model='stop_breakout_signal_bar', entry_params={'x_offset': 10.0},
    sl_method='signal_bar_atr_buffer', sl_params={'ksl_level': 2},
    tp_method='fib_atr_ctrader_v0', tp_params={'ktp_level': 6, 'exit_mode': 'fixed_only'},
    order_model='stop_order', order_params={'cancel_after_bars': 3},
    exit_model='fixed_only', exit_params={'sma_period': 20},
)

# Dotted names route values to entry/SL/TP/order/exit/risk components.
COARSE_GRID = {
    'entry.x_offset': [1.0, 10.0, 20.0, 30.0, 40.0],
    'stoploss.ksl_level': [1, 2, 3, 4],
    'takeprofit.ktp_level': [2, 4, 6, 8, 10, 12],
    'orders.cancel_after_bars': [2, 3, 4],
    'takeprofit.exit_mode': ['fixed_only'],
}
REFINEMENT_VALUES = {
    'entry.x_offset': [1.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0],
    'stoploss.ksl_level': [1, 2, 3, 4],
    'takeprofit.ktp_level': list(range(1, 13)),
    'orders.cancel_after_bars': [2, 3, 4],
    'takeprofit.exit_mode': ['fixed_only'],
}
FINE_TOP_N = 3
FINE_NEIGHBOR_STEPS = 1
FINE_MAX_RUNS = 250
METRIC_COL = 'expectancy_r'
NEIGHBORHOOD_PCT = 0.20
NEIGHBORHOOD_STEPS = 1
CANDIDATE_RULES = {
    'min_filled': 20,
    'min_fill_rate': 0.20,
    'max_pending_expired_rate': 0.80,
    'max_ambiguity_rate': 0.20,
    'min_local_count': 2,
}

MARKET_SPEC = MarketSpec(
    symbol=PREFERRED_SYMBOL, pip_size=1.0, pip_value_per_lot=1.0,
    min_lot=0.01, lot_step=0.01,
    commission_per_lot_per_side=0.0,
    spread_buffer_pips=0.0, slippage_buffer_pips=0.0,
)

signal_catalog = discover_signal_catalog(RAW_SIGNALS, SIGNAL_STRATEGY)
selected_signal = select_signal(
    signal_catalog, choice=SIGNAL_CHOICE,
    symbol=PREFERRED_SYMBOL, timeframe=PREFERRED_TIMEFRAME,
)
SIGNAL_FILE = selected_signal.path
SYMBOL = selected_signal.symbol
TIMEFRAME = selected_signal.timeframe
MARKET_SPEC = MarketSpec(**{**MARKET_SPEC.__dict__, 'symbol': SYMBOL})

display(Markdown('### Available Components'))
display(style_report(component_catalog_frame()))
display(Markdown('### Selected Controls'))
display(style_report(control_panel_frame(
    selection=selected_signal, run_config=BASE_CONFIG,
    market_spec=MARKET_SPEC, warmup_bars=WARMUP_BARS,
)))
display(Markdown('### Parameter Grid'))
grid_preview = pd.DataFrame({
    'parameter': list(COARSE_GRID),
    'coarse values': [str(v) for v in COARSE_GRID.values()],
    'fine value universe': [str(REFINEMENT_VALUES[key]) for key in COARSE_GRID],
})
display(grid_preview)
print('Coarse backtests:', grid_size(COARSE_GRID))
print('Fine backtests: calculated after coarse candidate selection; capped at', FINE_MAX_RUNS)

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)
start = signals['bartime'].min()
end = signals['bartime'].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(
    SYMBOL, TIMEFRAME, start=start, end=end,
    warmup_bars=WARMUP_BARS, tail_bars=5,
)
display(style_report(run_context_frame(
    signals=signals, bars=bars, selection=selected_signal,
    run_config=BASE_CONFIG,
)))

In [ ]:
def make_progress(label, every=10):
    handle = display(label + ': waiting...', display_id=True)
    def update(done, total, params):
        if done == 1 or done == total or done % every == 0:
            handle.update(label + f': {done}/{total} backtests completed')
    return update

coarse_map = evaluate_stability_map(
    signals=signals, bars=bars, symbol=SYMBOL, timeframe=TIMEFRAME,
    market_spec=MARKET_SPEC, base_config=BASE_CONFIG,
    param_grid=COARSE_GRID, metric_col=METRIC_COL,
    neighborhood_pct=NEIGHBORHOOD_PCT,
    neighborhood_steps=NEIGHBORHOOD_STEPS,
    progress_callback=make_progress('Coarse search'),
)
coarse_candidates = rank_stability_candidates(
    coarse_map, param_cols=list(COARSE_GRID),
    metric_col=METRIC_COL, **CANDIDATE_RULES,
)
FINE_GRID = build_refined_grid(
    coarse_candidates, REFINEMENT_VALUES,
    param_cols=list(COARSE_GRID), top_n=FINE_TOP_N,
    neighbor_steps=FINE_NEIGHBOR_STEPS,
    max_combinations=FINE_MAX_RUNS,
)
print('Fine grid:', FINE_GRID)
print('Fine backtests:', grid_size(FINE_GRID))
fine_map = evaluate_stability_map(
    signals=signals, bars=bars, symbol=SYMBOL, timeframe=TIMEFRAME,
    market_spec=MARKET_SPEC, base_config=BASE_CONFIG,
    param_grid=FINE_GRID, metric_col=METRIC_COL,
    neighborhood_pct=NEIGHBORHOOD_PCT,
    neighborhood_steps=NEIGHBORHOOD_STEPS,
    progress_callback=make_progress('Fine search'),
)
candidates = rank_stability_candidates(
    fine_map, param_cols=list(FINE_GRID),
    metric_col=METRIC_COL, **CANDIDATE_RULES,
)

display(Markdown('## Top Stable Candidates'))
display(candidates.head(30))
display(Markdown('## Coarse Search Leaders'))
display(coarse_candidates.head(20))
display(Markdown('## Fine Stability Map'))
display(fine_map.sort_values(
    METRIC_COL + '_stability_score', ascending=False,
).head(50))

In [ ]:
RUN_ID = make_run_id('stability', symbol=SYMBOL, timeframe=TIMEFRAME)
coarse_map_path = OUTPUT_DIR / (RUN_ID + '_coarse_map.csv')
coarse_candidates_path = OUTPUT_DIR / (RUN_ID + '_coarse_candidates.csv')
map_path = OUTPUT_DIR / (RUN_ID + '_fine_map.csv')
candidates_path = OUTPUT_DIR / (RUN_ID + '_candidates.csv')
coarse_map.to_csv(coarse_map_path, index=False)
coarse_candidates.to_csv(coarse_candidates_path, index=False)
fine_map.to_csv(map_path, index=False)
candidates.to_csv(candidates_path, index=False)

eligible_count = int(candidates['candidate_eligible'].sum()) if not candidates.empty else 0
best_score = candidates[METRIC_COL + '_stability_score'].max() if not candidates.empty else None
snapshot_path = save_snapshot(
    name=RUN_ID, run_id=RUN_ID, run_type='stability',
    config={
        'strategy': SIGNAL_STRATEGY, 'symbol': SYMBOL, 'timeframe': TIMEFRAME,
        'base_config': BASE_CONFIG, 'market_spec': MARKET_SPEC,
        'search_mode': 'coarse_to_fine',
        'param_grid': COARSE_GRID,
        'refinement_values': REFINEMENT_VALUES,
        'fine_top_n': FINE_TOP_N,
        'fine_neighbor_steps': FINE_NEIGHBOR_STEPS,
        'fine_max_combinations': FINE_MAX_RUNS,
        'metric_col': METRIC_COL,
        'neighborhood_pct': NEIGHBORHOOD_PCT,
        'neighborhood_steps': NEIGHBORHOOD_STEPS,
        'candidate_rules': CANDIDATE_RULES,
        'warmup_bars': WARMUP_BARS,
    },
    result_summary={
        'coarse_grid_rows': len(coarse_map),
        'fine_grid_rows': len(fine_map),
        'eligible_candidates': eligible_count,
        'best_stability_score': best_score,
    },
    outputs={
        'coarse_map': coarse_map_path,
        'coarse_candidates': coarse_candidates_path,
        'stability_map': map_path,
        'candidates': candidates_path,
    },
    signal_file=SIGNAL_FILE,
    market_data_source_id='core_python.data.loader',
    repo_root=project_root,
)
print('run_id:', RUN_ID)
print(coarse_map_path)
print(coarse_candidates_path)
print(map_path)
print(candidates_path)
print(snapshot_path)